# Step 5: Warp Tiling 优化

**Runtime -> Change runtime type -> GPU (T4)**

## 优化内容

### 核心思想: 三级 Tile 层次

v4 只有两级: Block Tile -> Thread Tile。线程到 C 元素的映射由 `(tx, ty)` 的 2D 网格决定，
warp 内的线程分布是隐式的，不一定是最优的。

v5 引入显式的 **Warp Tile** 层，形成三级结构:

```
Block Tile (128x128)
  -> Warp Tile (32x64)    -- 每个 warp 负责一块紧凑的 C 子区域
    -> Thread Tile (8x8)  -- 每个线程计算 8x8 个 C 元素

Warp 布局:           Thread 布局 (warp 内):
  BN=128              WN=64
  |--64--|--64--|      |--8--|--8--|...x8
  +------+------+ --  +-----+-----+...  --
  |warp0 |warp1 | 32  | t0  | t1  |...  8  (threadRowInWarp=0)
  +------+------+ --  +-----+-----+...  --
  |warp2 |warp3 | 32  | t8  | t9  |...  8  (threadRowInWarp=1)
  +------+------+ --  +-----+-----+...  --
  |warp4 |warp5 | 32  | t16 | t17 |...  8  (threadRowInWarp=2)
  +------+------+ --  +-----+-----+...  --
  |warp6 |warp7 | 32  | t24 | t25 |...  8  (threadRowInWarp=3)
  +------+------+ --  +-----+-----+...  --
  BM=128              WM=32
```

### 为什么 Warp 布局很重要?

**v4 的隐式 warp 映射** (2D block 16x16, tid = ty*16+tx):
- warp 0 = {ty=0, tx=0..15} + {ty=1, tx=0..15}
- warp tile = 2*TM x 16*TN = 16 x 128 — 非常扁平
- Bs 读取: 16 个不同 tx -> 16 个不同地址, 有 bank conflict

**v5 的显式 warp 映射** (1D block 256, warp/lane 显式计算):
- warp tile = WM x WN = 32 x 64 — 紧凑方形
- As 读取: 4 个 threadRowInWarp, 每个被 8 个 lane **广播** -> 0 bank conflict
- Bs 读取: 8 个 threadColInWarp, 每个被 4 个 lane **广播** -> 仅 8 个唯一地址
  (配合 swizzle -> 0 bank conflict)

### Shared Memory 访问量对比 (每 warp 每 bk 步)

| | v4 (隐式) | v5 (显式 warp tile) |
|---|---|---|
| As 唯一地址数 | 2 | 4 |
| Bs 唯一地址数 | 16 | 8 |
| **总唯一地址** | **18** | **12** (-33%) |
| As bank conflict | 0 (PAD) | 0 (PAD + 广播) |
| Bs bank conflict | ~2-way (swizzle) | 0 (swizzle + 广播) |

### 保留 v4 全部优化

- Double buffering (计算/加载重叠)
- Float4 vectorized store
- As PAD + Bs XOR swizzle

In [1]:
# 检查 GPU 和 CUDA 版本
!nvidia-smi
!nvcc --version

Mon Apr  6 09:02:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%writefile matmul_warp_tiling_v5.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define BM 128
#define BN 128
#define BK 8
#define TM 8
#define TN 8
#define WM 32          // warp tile M
#define WN 64          // warp tile N
#define WARP_SIZE 32
#define NUM_THREADS 256
#define NUM_WARPS_M (BM / WM)   // 4
#define NUM_WARPS_N (BN / WN)   // 2
#define THREAD_M (WM / TM)     // 4 (warp 内 M 方向线程数)
#define THREAD_N (WN / TN)     // 8 (warp 内 N 方向线程数)
#define A_PAD 1
#define BS_SWIZZLE(col) ((col) ^ (((col) >> 3) << 1))

// ============================================================
// v5 kernel: Warp Tiling + Double Buffering + Float4 Store
// ============================================================
// 在 v4 基础上引入显式 warp tile 层:
//   Block Tile (128x128) -> Warp Tile (32x64) -> Thread Tile (8x8)
//
// 使用 1D block (256 threads), 通过 warpId/laneId 显式映射:
//   warpId = tid / 32          (0..7)
//   warpRow = warpId / 2       (0..3, M 方向)
//   warpCol = warpId % 2       (0..1, N 方向)
//   threadRowInWarp = laneId / 8  (0..3)
//   threadColInWarp = laneId % 8  (0..7)
//
// 关键改善:
//   1. warp 内多个 lane 读同一 shared memory 地址 -> 硬件广播, 无 bank conflict
//   2. As: 8 个 lane 广播同一行 -> 4 个唯一地址/step
//   3. Bs: 4 个 lane 广播同一列 -> 8 个唯一地址/step (+ swizzle = 0 conflict)
//   4. warp tile 紧凑 (32x64) -> 更好的 L2 cache 局部性
// ============================================================
__global__ void MatMulWarpTilingKernel_v5(const float* __restrict__ A,
                                          const float* __restrict__ B,
                                          float* __restrict__ C,
                                          int M, int N, int K)
{
    const int bx = blockIdx.x;
    const int by = blockIdx.y;
    const int tid = threadIdx.x;  // 1D block: 0..255

    // ---- Warp/Lane 映射 ----
    const int warpId = tid / WARP_SIZE;           // 0..7
    const int laneId = tid % WARP_SIZE;           // 0..31
    const int warpRow = warpId / NUM_WARPS_N;     // 0..3 (M 方向)
    const int warpCol = warpId % NUM_WARPS_N;     // 0..1 (N 方向)
    const int threadRowInWarp = laneId / THREAD_N; // 0..3
    const int threadColInWarp = laneId % THREAD_N; // 0..7

    // ---- Double buffer shared memory ----
    __shared__ float As[2][BM][BK + A_PAD];  // 2 x 128 x 9
    __shared__ float Bs[2][BK][BN];          // 2 x 8 x 128

    float regC[TM][TN] = {0.0f};
    float regA[TM];
    float regB[TN];

    const int cRow = by * BM;
    const int cCol = bx * BN;
    const float* aPtr = A + cRow * K;
    const float* bPtr = B + cCol;

    // ---- 协作加载索引 (与 v4 相同, 基于 tid) ----
    const int loadARow = tid / BK;
    const int loadACol = tid % BK;
    const int strideA  = NUM_THREADS / BK;    // 32
    const int loadBRow = tid / BN;
    const int loadBCol = tid % BN;
    const int strideB  = NUM_THREADS / BN;    // 2

    // ========== 预取第一个 tile 到 buffer 0 ==========
    for (int i = 0; i < BM; i += strideA) {
        As[0][loadARow + i][loadACol] = aPtr[(loadARow + i) * K + loadACol];
    }
    for (int i = 0; i < BK; i += strideB) {
        Bs[0][loadBRow + i][BS_SWIZZLE(loadBCol)] = bPtr[(loadBRow + i) * N + loadBCol];
    }
    __syncthreads();

    // ========== 主循环: double buffering ==========
    int writeBuffer = 1;
    for (int k = BK; k < K; k += BK) {
        int readBuffer = 1 - writeBuffer;

        // 加载下一个 tile 到 writeBuffer
        for (int i = 0; i < BM; i += strideA) {
            As[writeBuffer][loadARow + i][loadACol] =
                aPtr[(loadARow + i) * K + k + loadACol];
        }
        for (int i = 0; i < BK; i += strideB) {
            Bs[writeBuffer][loadBRow + i][BS_SWIZZLE(loadBCol)] =
                bPtr[(k + loadBRow + i) * N + loadBCol];
        }

        // ---- 从 readBuffer 计算: 使用 warp tile 映射 ----
        // As 读取: As[warpRow*WM + threadRowInWarp*TM + tm][bk]
        //   同一 warpRow+threadRowInWarp 的 8 个 lane 读同一地址 -> 广播
        // Bs 读取: Bs[bk][warpCol*WN + threadColInWarp*TN + tn]
        //   同一 warpCol+threadColInWarp 的 4 个 lane 读同一地址 -> 广播
        for (int bk = 0; bk < BK; ++bk) {
            for (int tm = 0; tm < TM; ++tm)
                regA[tm] = As[readBuffer][warpRow * WM + threadRowInWarp * TM + tm][bk];
            for (int tn = 0; tn < TN; ++tn)
                regB[tn] = Bs[readBuffer][bk][BS_SWIZZLE(warpCol * WN + threadColInWarp * TN + tn)];
            for (int tm = 0; tm < TM; ++tm)
                for (int tn = 0; tn < TN; ++tn)
                    regC[tm][tn] += regA[tm] * regB[tn];
        }

        writeBuffer = 1 - writeBuffer;
        __syncthreads();
    }

    // ========== 计算最后一个 tile ==========
    {
        int readBuffer = 1 - writeBuffer;
        for (int bk = 0; bk < BK; ++bk) {
            for (int tm = 0; tm < TM; ++tm)
                regA[tm] = As[readBuffer][warpRow * WM + threadRowInWarp * TM + tm][bk];
            for (int tn = 0; tn < TN; ++tn)
                regB[tn] = Bs[readBuffer][bk][BS_SWIZZLE(warpCol * WN + threadColInWarp * TN + tn)];
            for (int tm = 0; tm < TM; ++tm)
                for (int tn = 0; tn < TN; ++tn)
                    regC[tm][tn] += regA[tm] * regB[tn];
        }
    }

    // ========== Float4 向量化写回 C ==========
    // warpCol*WN + threadColInWarp*TN 是 8 的倍数 -> 32B 对齐 > 16B 要求
    for (int tm = 0; tm < TM; ++tm) {
        int globalRow = cRow + warpRow * WM + threadRowInWarp * TM + tm;
        int globalCol = cCol + warpCol * WN + threadColInWarp * TN;
        reinterpret_cast<float4*>(&C[globalRow * N + globalCol])[0] =
            make_float4(regC[tm][0], regC[tm][1], regC[tm][2], regC[tm][3]);
        reinterpret_cast<float4*>(&C[globalRow * N + globalCol + 4])[0] =
            make_float4(regC[tm][4], regC[tm][5], regC[tm][6], regC[tm][7]);
    }
}

// ============================================================
// v4 kernel: 用于对比 (double buffer + float4 store, 无 warp tiling)
// ============================================================
__global__ void MatMulRegTilingKernel_v4(const float* __restrict__ A,
                                         const float* __restrict__ B,
                                         float* __restrict__ C,
                                         int M, int N, int K)
{
    const int bx = blockIdx.x;
    const int by = blockIdx.y;
    const int tx = threadIdx.x;
    const int ty = threadIdx.y;
    const int tid = ty * blockDim.x + tx;
    const int numThreads = blockDim.x * blockDim.y;

    __shared__ float As[2][BM][BK + A_PAD];
    __shared__ float Bs[2][BK][BN];

    float regC[TM][TN] = {0.0f};
    float regA[TM];
    float regB[TN];

    const int cRow = by * BM;
    const int cCol = bx * BN;
    const float* aPtr = A + cRow * K;
    const float* bPtr = B + cCol;

    const int loadARow = tid / BK;
    const int loadACol = tid % BK;
    const int strideA  = numThreads / BK;
    const int loadBRow = tid / BN;
    const int loadBCol = tid % BN;
    const int strideB  = numThreads / BN;

    for (int i = 0; i < BM; i += strideA)
        As[0][loadARow + i][loadACol] = aPtr[(loadARow + i) * K + loadACol];
    for (int i = 0; i < BK; i += strideB)
        Bs[0][loadBRow + i][BS_SWIZZLE(loadBCol)] = bPtr[(loadBRow + i) * N + loadBCol];
    __syncthreads();

    int writeBuffer = 1;
    for (int k = BK; k < K; k += BK) {
        int readBuffer = 1 - writeBuffer;
        for (int i = 0; i < BM; i += strideA)
            As[writeBuffer][loadARow + i][loadACol] =
                aPtr[(loadARow + i) * K + k + loadACol];
        for (int i = 0; i < BK; i += strideB)
            Bs[writeBuffer][loadBRow + i][BS_SWIZZLE(loadBCol)] =
                bPtr[(k + loadBRow + i) * N + loadBCol];
        for (int bk = 0; bk < BK; ++bk) {
            for (int tm = 0; tm < TM; ++tm)
                regA[tm] = As[readBuffer][ty * TM + tm][bk];
            for (int tn = 0; tn < TN; ++tn)
                regB[tn] = Bs[readBuffer][bk][BS_SWIZZLE(tx * TN + tn)];
            for (int tm = 0; tm < TM; ++tm)
                for (int tn = 0; tn < TN; ++tn)
                    regC[tm][tn] += regA[tm] * regB[tn];
        }
        writeBuffer = 1 - writeBuffer;
        __syncthreads();
    }
    {
        int readBuffer = 1 - writeBuffer;
        for (int bk = 0; bk < BK; ++bk) {
            for (int tm = 0; tm < TM; ++tm)
                regA[tm] = As[readBuffer][ty * TM + tm][bk];
            for (int tn = 0; tn < TN; ++tn)
                regB[tn] = Bs[readBuffer][bk][BS_SWIZZLE(tx * TN + tn)];
            for (int tm = 0; tm < TM; ++tm)
                for (int tn = 0; tn < TN; ++tn)
                    regC[tm][tn] += regA[tm] * regB[tn];
        }
    }

    for (int tm = 0; tm < TM; ++tm) {
        int globalRow = cRow + ty * TM + tm;
        reinterpret_cast<float4*>(&C[globalRow * N + cCol + tx * TN])[0] =
            make_float4(regC[tm][0], regC[tm][1], regC[tm][2], regC[tm][3]);
        reinterpret_cast<float4*>(&C[globalRow * N + cCol + tx * TN + 4])[0] =
            make_float4(regC[tm][4], regC[tm][5], regC[tm][6], regC[tm][7]);
    }
}

int main()
{
    int M = 2048, N = 2048, K = 2048;

    size_t bytesA = M * K * sizeof(float);
    size_t bytesB = K * N * sizeof(float);
    size_t bytesC = M * N * sizeof(float);

    float* A     = (float*)malloc(bytesA);
    float* B     = (float*)malloc(bytesB);
    float* C_v4  = (float*)malloc(bytesC);
    float* C_v5  = (float*)malloc(bytesC);

    for (int i = 0; i < M * K; ++i) A[i] = (float)(i % 10) * 0.1f;
    for (int i = 0; i < K * N; ++i) B[i] = (float)(i % 7)  * 0.1f;

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, bytesA);
    cudaMalloc(&d_B, bytesB);
    cudaMalloc(&d_C, bytesC);
    cudaMemcpy(d_A, A, bytesA, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B, bytesB, cudaMemcpyHostToDevice);

    // v4: 2D block (16, 16)
    dim3 dimBlock_v4(BN / TN, BM / TM);  // (16, 16)
    dim3 dimGrid(N / BN, M / BM);        // (16, 16)

    // v5: 1D block (256)
    dim3 dimBlock_v5(NUM_THREADS);        // (256)

    cudaFuncSetCacheConfig(MatMulRegTilingKernel_v4, cudaFuncCachePreferShared);
    cudaFuncSetCacheConfig(MatMulWarpTilingKernel_v5, cudaFuncCachePreferShared);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    float ms = 0;

    // ========== v4 baseline ==========
    MatMulRegTilingKernel_v4<<<dimGrid, dimBlock_v4>>>(d_A, d_B, d_C, M, N, K);
    cudaDeviceSynchronize();
    cudaEventRecord(start);
    MatMulRegTilingKernel_v4<<<dimGrid, dimBlock_v4>>>(d_A, d_B, d_C, M, N, K);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&ms, start, stop);
    float v4_ms = ms;
    cudaMemcpy(C_v4, d_C, bytesC, cudaMemcpyDeviceToHost);

    // ========== v5 warp tiling ==========
    MatMulWarpTilingKernel_v5<<<dimGrid, dimBlock_v5>>>(d_A, d_B, d_C, M, N, K);
    cudaDeviceSynchronize();
    cudaEventRecord(start);
    MatMulWarpTilingKernel_v5<<<dimGrid, dimBlock_v5>>>(d_A, d_B, d_C, M, N, K);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&ms, start, stop);
    float v5_ms = ms;
    cudaMemcpy(C_v5, d_C, bytesC, cudaMemcpyDeviceToHost);

    // ========== 验证 ==========
    float max_err = 0.0f;
    for (int i = 0; i < M * N; ++i) {
        float err = fabsf(C_v4[i] - C_v5[i]);
        if (err > max_err) max_err = err;
    }

    double gflops_v4 = 2.0 * M * N * K / (v4_ms * 1e-3) / 1e9;
    double gflops_v5 = 2.0 * M * N * K / (v5_ms * 1e-3) / 1e9;

    printf("=== GEMM: v4 vs v5 (Warp Tiling) ===\n");
    printf("Matrix: %dx%dx%d\n", M, N, K);
    printf("Block tile: %dx%d, Warp tile: %dx%d, Thread tile: %dx%d, K tile: %d\n",
           BM, BN, WM, WN, TM, TN, BK);
    printf("\n");
    printf("v4 (dbuf+f4st):            %.3f ms  (%.1f GFLOPS)\n", v4_ms, gflops_v4);
    printf("v5 (+ warp tiling):        %.3f ms  (%.1f GFLOPS)\n", v5_ms, gflops_v5);
    printf("Speedup v5/v4: %.2fx\n", v4_ms / v5_ms);
    printf("Max diff (v4 vs v5): %e\n", max_err);
    if (max_err < 1e-3f)
        printf("PASSED\n");
    else
        printf("FAILED\n");

    cudaEventDestroy(start); cudaEventDestroy(stop);
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(A); free(B); free(C_v4); free(C_v5);
    return 0;
}


Writing matmul_warp_tiling_v5.cu


In [3]:
!nvcc -O3 -lineinfo -arch=sm_75 matmul_warp_tiling_v5.cu -o matmul_warp_tiling_v5
print("编译成功!")

编译成功!


In [4]:
# 运行对比测试
!./matmul_warp_tiling_v5

=== GEMM: v4 vs v5 (Warp Tiling) ===
Matrix: 2048x2048x2048
Block tile: 128x128, Warp tile: 32x64, Thread tile: 8x8, K tile: 8

v4 (dbuf+f4st):            9.523 ms  (1804.0 GFLOPS)
v5 (+ warp tiling):        9.713 ms  (1768.8 GFLOPS)
Speedup v5/v4: 0.98x
Max diff (v4 vs v5): 0.000000e+00
PASSED


---
## Part 1: Nsight Systems

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!chmod -R +x /content/drive/MyDrive/cuda/nsys_tool

NSYS = "/content/drive/MyDrive/cuda/nsys_tool/target-linux-x64/nsys"
!{NSYS} --version

Mounted at /content/drive
NVIDIA Nsight Systems version 2024.5.1.113-245134619542v0


In [6]:
NSYS = "/content/drive/MyDrive/cuda/nsys_tool/target-linux-x64/nsys"
!{NSYS} profile \
     --stats=true \
     --force-overwrite=true \
     -o nsys_warp_tiling_v5 \
     ./matmul_warp_tiling_v5

=== GEMM: v4 vs v5 (Warp Tiling) ===
Matrix: 2048x2048x2048
Block tile: 128x128, Warp tile: 32x64, Thread tile: 8x8, K tile: 8

v4 (dbuf+f4st):            9.781 ms  (1756.5 GFLOPS)
v5 (+ warp tiling):        9.981 ms  (1721.2 GFLOPS)
Speedup v5/v4: 0.98x
Max diff (v4 vs v5): 0.000000e+00
PASSED
Generating '/tmp/nsys-report-342d.qdstrm'
[1/8] [========================100%] nsys_warp_tiling_v5.nsys-rep
[2/8] [========================100%] nsys_warp_tiling_v5.sqlite
[3/8] Executing 'nvtx_sum' stats report
SKIPPED: /content/nsys_warp_tiling_v5.sqlite does not contain NV Tools Extension (NVTX) data.
[4/8] Executing 'osrt_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)     Med (ns)    Min (ns)   Max (ns)    StdDev (ns)            Name         
 --------  ---------------  ---------  ------------  -----------  --------  -----------  ------------  ----------------------
     74.5      557,453,151         14  39,818,082.2  8,372,509.5     1,238  356,240,164  93,358,707.1  po

In [7]:
NSYS = "/content/drive/MyDrive/cuda/nsys_tool/target-linux-x64/nsys"
!{NSYS} stats --force-export=true --report cuda_gpu_kern_sum nsys_warp_tiling_v5.nsys-rep

Generating SQLite file nsys_warp_tiling_v5.sqlite from nsys_warp_tiling_v5.nsys-rep
Processing [nsys_warp_tiling_v5.sqlite] with [/content/drive/MyDrive/cuda/nsys_tool/host-linux-x64/reports/cuda_gpu_kern_sum.py]... 

 ** CUDA GPU Kernel Summary (cuda_gpu_kern_sum):

 Time (%)  Total Time (ns)  Instances   Avg (ns)     Med (ns)    Min (ns)   Max (ns)   StdDev (ns)                                       Name                                      
 --------  ---------------  ---------  -----------  -----------  ---------  ---------  -----------  -------------------------------------------------------------------------------
     50.5       19,915,277          2  9,957,638.5  9,957,638.5  9,952,775  9,962,502      6,878.0  MatMulWarpTilingKernel_v5(const float *, const float *, float *, int, int, int)
     49.5       19,503,125          2  9,751,562.5  9,751,562.5  9,740,491  9,762,634     15,657.5  MatMulRegTilingKernel_v4(const float *, const float *, float *, int, int, int) 



---
## Part 2: Nsight Compute — v5 深度分析

重点验证:
1. shared memory bank conflict 是否进一步降低 (广播效果)
2. warp stall / eligible warps 是否改善
3. compute throughput 是否提升
4. warp tile 的 L2 cache 局部性效果

In [8]:
# Memory Workload
!ncu --section MemoryWorkloadAnalysis \
     --section MemoryWorkloadAnalysis_Tables \
     --kernel-name MatMulWarpTilingKernel_v5 \
     --launch-skip 1 --launch-count 1 \
     ./matmul_warp_tiling_v5

==PROF== Connected to process 1542 (/content/matmul_warp_tiling_v5)
==PROF== Profiling "MatMulWarpTilingKernel_v5": 0%....50%....100% - 27 passes
=== GEMM: v4 vs v5 (Warp Tiling) ===
Matrix: 2048x2048x2048
Block tile: 128x128, Warp tile: 32x64, Thread tile: 8x8, K tile: 8

v4 (dbuf+f4st):            6.806 ms  (2524.4 GFLOPS)
v5 (+ warp tiling):        5970.081 ms  (2.9 GFLOPS)
Speedup v5/v4: 0.00x
Max diff (v4 vs v5): 0.000000e+00
PASSED
==PROF== Disconnected from process 1542
[1542] matmul_warp_tiling_v5@127.0.0.1
  MatMulWarpTilingKernel_v5(const float *, const float *, float *, int, int, int) (16, 16, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Memory Workload Analysis
    ----------------- ----------- ------------
    Metric Name       Metric Unit Metric Value
    ----------------- ----------- ------------
    Memory Throughput     Gbyte/s        22.28
    Mem Busy                    %        29.94
    Max Bandwidth               %        44.23
    L1/TEX Hit

In [9]:
# Occupancy + Launch Stats
!ncu --section Occupancy \
     --section LaunchStats \
     --kernel-name MatMulWarpTilingKernel_v5 \
     --launch-skip 1 --launch-count 1 \
     ./matmul_warp_tiling_v5

==PROF== Connected to process 1636 (/content/matmul_warp_tiling_v5)
==PROF== Profiling "MatMulWarpTilingKernel_v5": 0%....50%....100% - 1 pass
=== GEMM: v4 vs v5 (Warp Tiling) ===
Matrix: 2048x2048x2048
Block tile: 128x128, Warp tile: 32x64, Thread tile: 8x8, K tile: 8

v4 (dbuf+f4st):            4.344 ms  (3955.0 GFLOPS)
v5 (+ warp tiling):        771.609 ms  (22.3 GFLOPS)
Speedup v5/v4: 0.01x
Max diff (v4 vs v5): 0.000000e+00
PASSED
==PROF== Disconnected from process 1636
[1636] matmul_warp_tiling_v5@127.0.0.1
  MatMulWarpTilingKernel_v5(const float *, const float *, float *, int, int, int) (16, 16, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Launch Statistics
    -------------------------------- --------------- -----------------
    Metric Name                          Metric Unit      Metric Value
    -------------------------------- --------------- -----------------
    Block Size                                                     256
    Function Cache Con

In [10]:
# Scheduler + Warp Stall
!ncu --section SchedulerStats \
     --section WarpStateStats \
     --kernel-name MatMulWarpTilingKernel_v5 \
     --launch-skip 1 --launch-count 1 \
     ./matmul_warp_tiling_v5

==PROF== Connected to process 1675 (/content/matmul_warp_tiling_v5)
==PROF== Profiling "MatMulWarpTilingKernel_v5": 0%....50%....100% - 8 passes
=== GEMM: v4 vs v5 (Warp Tiling) ===
Matrix: 2048x2048x2048
Block tile: 128x128, Warp tile: 32x64, Thread tile: 8x8, K tile: 8

v4 (dbuf+f4st):            5.913 ms  (2905.6 GFLOPS)
v5 (+ warp tiling):        2388.999 ms  (7.2 GFLOPS)
Speedup v5/v4: 0.00x
Max diff (v4 vs v5): 0.000000e+00
PASSED
==PROF== Disconnected from process 1675
[1675] matmul_warp_tiling_v5@127.0.0.1
  MatMulWarpTilingKernel_v5(const float *, const float *, float *, int, int, int) (16, 16, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Scheduler Statistics
    ---------------------------- ----------- ------------
    Metric Name                  Metric Unit Metric Value
    ---------------------------- ----------- ------------
    One or More Eligible                   %        41.90
    Issued Warp Per Scheduler                        0.42
    No Elig

In [11]:
# Speed of Light + Compute
!ncu --section SpeedOfLight \
     --section ComputeWorkloadAnalysis \
     --kernel-name MatMulWarpTilingKernel_v5 \
     --launch-skip 1 --launch-count 1 \
     ./matmul_warp_tiling_v5

==PROF== Connected to process 1729 (/content/matmul_warp_tiling_v5)
==PROF== Profiling "MatMulWarpTilingKernel_v5": 0%....50%....100% - 8 passes
=== GEMM: v4 vs v5 (Warp Tiling) ===
Matrix: 2048x2048x2048
Block tile: 128x128, Warp tile: 32x64, Thread tile: 8x8, K tile: 8

v4 (dbuf+f4st):            6.238 ms  (2754.0 GFLOPS)
v5 (+ warp tiling):        2576.459 ms  (6.7 GFLOPS)
Speedup v5/v4: 0.00x
Max diff (v4 vs v5): 0.000000e+00
PASSED
==PROF== Disconnected from process 1729
[1729] matmul_warp_tiling_v5@127.0.0.1
  MatMulWarpTilingKernel_v5(const float *, const float *, float *, int, int, int) (16, 16, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         5.00
    SM Frequency                    Mhz       585.00
    Elapsed Cycles         

In [12]:
# Source Counters
!ncu --section SourceCounters \
     --kernel-name MatMulWarpTilingKernel_v5 \
     --launch-skip 1 --launch-count 1 \
     ./matmul_warp_tiling_v5

==PROF== Connected to process 1793 (/content/matmul_warp_tiling_v5)
==PROF== Profiling "MatMulWarpTilingKernel_v5": 0%....50%....100% - 5 passes
=== GEMM: v4 vs v5 (Warp Tiling) ===
Matrix: 2048x2048x2048
Block tile: 128x128, Warp tile: 32x64, Thread tile: 8x8, K tile: 8

v4 (dbuf+f4st):            7.917 ms  (2170.0 GFLOPS)
v5 (+ warp tiling):        2565.002 ms  (6.7 GFLOPS)
Speedup v5/v4: 0.00x
Max diff (v4 vs v5): 0.000000e+00
PASSED
==PROF== Disconnected from process 1793
[1793] matmul_warp_tiling_v5@127.0.0.1
  MatMulWarpTilingKernel_v5(const float *, const float *, float *, int, int, int) (16, 16, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Source Counters
    ------------------------- ----------- ------------
    Metric Name               Metric Unit Metric Value
    ------------------------- ----------- ------------
    Branch Instructions Ratio           %         0.00
    Branch Instructions              inst      542,720
    Branch Efficiency          

In [13]:
# 导出完整报告
!ncu --set full \
     --kernel-name MatMulWarpTilingKernel_v5 \
     --launch-skip 1 --launch-count 1 \
     -o warp_tiling_v5_profile \
     ./matmul_warp_tiling_v5

print("\n报告已保存为 warp_tiling_v5_profile.ncu-rep")

==PROF== Connected to process 1863 (/content/matmul_warp_tiling_v5)
==PROF== Profiling "MatMulWarpTilingKernel_v5": 0%....50%....100% - 31 passes
=== GEMM: v4 vs v5 (Warp Tiling) ===
Matrix: 2048x2048x2048
Block tile: 128x128, Warp tile: 32x64, Thread tile: 8x8, K tile: 8

v4 (dbuf+f4st):            6.056 ms  (2837.0 GFLOPS)
v5 (+ warp tiling):        9071.490 ms  (1.9 GFLOPS)
Speedup v5/v4: 0.00x
Max diff (v4 vs v5): 0.000000e+00
PASSED
==PROF== Disconnected from process 1863
==PROF== Report: /content/warp_tiling_v5_profile.ncu-rep

报告已保存为 warp_tiling_v5_profile.ncu-rep


In [14]:
from google.colab import files
files.download('warp_tiling_v5_profile.ncu-rep')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## 对比检查清单

| 指标 | v1 (原版) | v3 (PAD+swizzle) | v4 (+ dbuf + f4st) | v5 (+ warp tile) | 变化 |
|------|-----------|-------------------|---------------------|-------------------|------|
| Kernel 时间 (ms) | 8.378 | 5.713 | | | |
| GFLOPS | 2050.5 | 3007.2 | | | |
| Compute Throughput | 40.68% | 64.04% | | | |
| Memory Throughput | 43.04% | 64.04% | | | |
| IPC Active | 1.25 | 1.92 | | | |
| Warp Cycles/Instruction | 12.22 | 7.74 | | | |
| Eligible Warps/Scheduler | 0.42 | 1.08 | | | |
| Shared Load bank conflict | 3.2-way | ~1.2-way | | | |
| Global store util | 4.0/32 | 4.0/32 | | | |
| Shared Mem/Block (KB) | 8.19 | 8.70 | ~17.4 | ~17.4 | |
| Registers/Thread | 96 | 128 | | | |
| Occupancy | 50% | 50% | | | |

### Warp Tiling 的 Bank Conflict 分析

**As 读取** `As[warpRow*32 + threadRowInWarp*8 + tm][bk]`:
- warp 内 32 个 lane 只有 4 个唯一的 threadRowInWarp (0,1,2,3)
- 每 8 个 lane 读同一地址 -> **硬件广播, 0 bank conflict**
- PAD=1 保证 4 个唯一地址在不同 bank

**Bs 读取** `Bs[bk][warpCol*64 + threadColInWarp*8 + tn]`:
- 32 个 lane 只有 8 个唯一的 threadColInWarp (0..7)
- 每 4 个 lane 读同一地址 -> **广播**
- 8 个唯一地址的 bank (无 swizzle): {0,8,16,24,0,8,16,24} -> 2-way
- 8 个唯一地址的 bank (有 swizzle): {0,10,20,30,8,2,28,14} -> **0 bank conflict**